# 🚗 Indian ANPR System - Free Google Colab Backend Server
**Hardware:** Google Cloud (12.7 GB RAM + Free GPU/CPU)  
**Status:** 100% Free, Zero Out-Of-Memory Crashes  
---
### How It Works:
1. **Render** hosts your fast, beautiful React frontend (100% free, 0 timeout).
2. **Google Colab** runs your AI backend (YOLOv8 + EasyOCR + Flask + 12.7GB RAM).
3. Run this notebook, copy the **Backend URL**, and paste it into your Render site via the **⚙️ Change** button!

In [ ]:
# Step 1: Clone your GitHub repository
# Change this to your GitHub repository URL if different:
REPO_URL = "https://github.com/amithneranjane/ANPR_FINAL.git"

import os, subprocess, time, re
!rm -rf anpr
!git clone $REPO_URL anpr
%cd anpr

# Step 2: Install required packages (CPU PyTorch, YOLOv8, EasyOCR, Flask)
print("📦 Installing dependencies (takes ~1 minute)...")
!pip install -q -r requirements.txt

# Step 3: Install Cloudflare Tunnel for direct HTTPS access (Fast & No password needed)
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared

# Step 4: Start Flask Backend Server in background
print("🚀 Starting ANPR Backend Server...")
flask_proc = subprocess.Popen(["python", "app.py"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(6)

# Step 5: Start Cloudflare Tunnel and display public URL
print("🌐 Connecting to Cloudflare Global Edge Tunnel...")
cf_proc = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

public_url = None
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")

for _ in range(40):
    line = cf_proc.stdout.readline()
    if not line: continue
    m = url_pattern.search(line)
    if m:
        public_url = m.group(0)
        break
    time.sleep(0.5)

print("=" * 60)
if public_url:
    print("🎉 YOUR ANPR BACKEND SERVER IS LIVE ONLINE!")
    print(f"🔗 BACKEND URL: {public_url}")
    print("=" * 60)
    print("👉 HOW TO CONNECT TO YOUR RENDER WEBSITE:")
    print(f"1. Copy this URL: {public_url}")
    print("2. Open your Render website (e.g. https://your-site.onrender.com)")
    print("3. Click the '⚙️ Change' button at the bottom-right corner.")
    print("4. Paste the URL and click 'Save & Connect'. Done!")
    print("=" * 60)
    print("ℹ️ Alternatively, you can also use this link directly as a standalone app!")
else:
    print("Cloudflare tunnel busy. Starting Localtunnel fallback:")
    !npx -y localtunnel --port 5000
